# 05 · Calibration Analysis

Reads `runs/final_eval/calibration.json` (produced by `scripts/fit_calibration.py`) and renders:
- Before/after ECE, adaptive-ECE, MCE, Brier, NLL per detector + fusion.
- Reliability diagrams (10-bin) for each configuration.

Regenerate the source JSON first: `python scripts/fit_calibration.py --split val`.

In [ ]:
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
REPORT = ROOT / 'runs' / 'final_eval' / 'calibration.json'
assert REPORT.exists(), f'missing {REPORT}: run scripts/fit_calibration.py first'
report = json.loads(REPORT.read_text())
list(report)

In [ ]:
import pandas as pd
rows = []
for det in ('yolo', 'rfdetr', 'fusion'):
    d = report.get(det, {})
    T = d.get('T')
    for stage in ('before', 'after'):
        s = d.get(stage, {})
        if not s:
            continue
        rows.append({
            'detector': det, 'stage': stage, 'T': T, 'n': s.get('n'),
            'ECE': s.get('ece'), 'aECE': s.get('adaptive_ece'),
            'MCE': s.get('mce'), 'Brier': s.get('brier'), 'NLL': s.get('nll'),
        })
pd.DataFrame(rows).round(4)

In [ ]:
def plot_reliability(ax, bins, title):
    xs = [(b['lo'] + b['hi']) / 2 for b in bins]
    ys = [b['empirical_acc'] for b in bins]
    counts = [b['count'] for b in bins]
    ax.plot([0, 1], [0, 1], '--', color='0.6', linewidth=1)
    total = max(1, sum(counts))
    alphas = [0.25 + 0.75 * c / total for c in counts]
    for x, y, a in zip(xs, ys, alphas):
        ax.bar(x, y, width=(bins[0]['hi'] - bins[0]['lo']) * 0.9,
               color='#1F5C4C', alpha=a, edgecolor='none')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xlabel('confidence'); ax.set_ylabel('empirical acc')
    ax.set_title(title, fontsize=10)

cfgs = [(d, s) for d in ('yolo', 'rfdetr', 'fusion')
         for s in ('before', 'after')
         if report.get(d, {}).get(s)]
cols = 2
rows = (len(cfgs) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(9, 3.2 * rows))
axes = axes.ravel() if rows * cols > 1 else [axes]
for ax, (det, stage) in zip(axes, cfgs):
    bins = report[det][stage].get('reliability', [])
    plot_reliability(ax, bins, f'{det} · {stage}  (ECE={report[det][stage]["ece"]:.4f})')
for ax in axes[len(cfgs):]:
    ax.axis('off')
plt.tight_layout(); plt.show()